# People Analytics & Recrutamento: Matching de Perfis do LinkedIn com Vagas

**Contexto de negócio**

Em processos seletivos com centenas ou milhares de candidatos para uma única vaga, a triagem manual de currículos e perfis do LinkedIn consome dezenas de horas da equipe de recrutamento. Além disso, triagens manuais sob pressão de tempo tendem a sofrer com inconsistência e viés.

O objetivo desta aplicação é calcular automaticamente um **Score de Aderência (Match Score de 0 a 100%)** entre o perfil do candidato no LinkedIn (resumo profissional, experiências, habilidades e tempo de carreira) e a **descrição das exigências da vaga**.

**1. Por que uma rede neural traz vantagens sobre buscas por palavras-chave?**

- **Diferença de vocabulário:** O candidato pode escrever *'Engenheiro de Dados'* e *'Pipeline de ETL em Python'*, enquanto a vaga pede *'Especialista em Big Data'* e *'Arquitetura de Dados'*. Uma busca tradicional por palavra-chave (TF-IDF / sintática) considera essas expressões diferentes.
- **Interações não lineares:** A aderência real não depende apenas de bater palavras-chave, mas da **combinação** entre o alinhamento semântico do histórico profissional e o atendimento a requisitos críticos (como senioridade e tempo de experiência). Redes neurais (MLP sobre representações vetoriais) aprendem a ponderar essa relação de forma mais flexível.

**0. Instalação de dependências**

Este notebook usa `numpy`, `pandas`, `scikit-learn` e `matplotlib`.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
print('Dependencias OK.')

Dependencias OK.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

**2. Geração do dataset simulado**

Simulamos 600 candidatos aplicando para uma vaga de **Cientista de Dados / Engenheiro de IA**. 
Cada candidato possui:
- `texto_linkedin`: Resumo do perfil + experiências listadas.
- `anos_experiencia`: Anos de experiência acumulados na carreira.
- `vaga_descricao`: Descrição textual dos requisitos da vaga (exigências de IA, Python, SQL, Machine Learning, MLOps).

O `match_score_real` (0 a 100) reflete a aderência verdadeira considerando tanto a similaridade semântica real quanto a senioridade.

In [ ]:
n = 600

vaga_texto = (
    'Buscamos Cientista de Dados com experiencia em Python, Machine Learning, '
    'Redes Neurais, SQL, MLOps e modelagem preditiva em producao para People Analytics e negocios.'
)
vaga_exp_exigida = 4.0

perfis_base = [
    'Cientista de dados com foco em Python, Machine Learning, modelos preditivos e SQL avançado em empresas de tecnologia.',
    'Especialista em Inteligência Artificial, Deep Learning, PyTorch, TensorFlow e MLOps para implantação de redes neurais.',
    'Analista de Dados com experiência em SQL, PowerBI, Excel avançado e dashboards de negócios para área comercial.',
    'Desenvolvedor Software Backend Python, API REST, Docker, PostgreSQL e microsserviços cloud.',
    'Engenheiro de Machine Learning com vivência em MLOps, infraestrutura de modelos em nuvem, Python e CI/CD.',
    'Estagiário de Administração apaixonado por estatística, análise de dados e suporte ao time de vendas.',
    'Pesquisador Acadêmico em Estatística e Aprendizado de Máquina com foco em redes neurais profundas e publicação de artigos.',
    'Gerente de Produtos de Dados com visão de negócios, People Analytics, liderança de equipes e métodos ágeis.'
]

indices = np.random.choice(len(perfis_base), size=n)
textos_candidatos = [perfis_base[i] for i in indices]

# Adiciona variação aleatória de vocabulário aos resumos
ruidos_texto = [
    ' Forte atuação em projetos de grande impacto.',
    ' Apaixonado por resolução de problemas complexos.',
    ' Experiência em ambiente corporativo e startups.',
    ' Foco em entregas ágeis e colaboração em equipe.'
]
textos_candidatos = [t + np.random.choice(ruidos_texto) for t in textos_candidatos]

anos_experiencia = np.round(np.clip(np.random.gamma(shape=2.5, scale=1.8, size=n), 0.5, 20.0), 1)

# Construção do ground truth (Match Score Real)
# 1. Qualidade técnica do texto (pesos por perfil)
pesos_tecnicos = [92, 96, 58, 62, 90, 30, 85, 68]
score_tecnico_base = np.array([pesos_tecnicos[i] for i in indices])

# 2. Fator de senioridade/experiência (penaliza se < exigido, recompensa suavemente se >=)
fator_exp = np.where(
    anos_experiencia < vaga_exp_exigida,
    (anos_experiencia / vaga_exp_exigida) ** 1.5,
    1.0 + 0.05 * np.log1p(np.maximum(0, anos_experiencia - vaga_exp_exigida))
)

ruido = np.random.normal(0, 4.5, size=n)
match_score_real = np.clip(score_tecnico_base * fator_exp + ruido, 0, 100)

df = pd.DataFrame({
    'texto_linkedin': textos_candidatos,
    'anos_experiencia': anos_experiencia,
    'vaga_descricao': [vaga_texto] * n,
    'match_score_real': match_score_real
})

print(f'Total de candidatos simulados: {len(df)}')
print(f'Match Score médio: {df["match_score_real"].mean():.1f}%')
df.head()

Total de candidatos simulados: 600
Match Score médio: 55.2%


**3. Separação em Treino e Teste**

Dividimos 70% dos dados para treino e 30% para teste.

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    df[['texto_linkedin', 'anos_experiencia']],
    df['match_score_real'],
    test_size=0.3,
    random_state=42
)

print(f'Treino: {len(X_train_df)} candidatos | Teste: {len(X_test_df)} candidatos')

Treino: 420 candidatos | Teste: 180 candidatos


**4. Modelo Baseline: TF-IDF + Similaridade de Cosseno + Regra de Experiência**

O baseline calcula a similaridade vetorial de palavras entre o perfil e a vaga via TF-IDF e multiplica por um fator proporcional aos anos de experiência.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2))
# Ajusta no texto da vaga + treinos do LinkedIn
tfidf.fit([vaga_texto] + list(X_train_df['texto_linkedin']))

vaga_vec = tfidf.transform([vaga_texto])

# Predição Baseline no Teste
test_linkedin_vec = tfidf.transform(X_test_df['texto_linkedin'])
sim_cosseno_test = cosine_similarity(test_linkedin_vec, vaga_vec).flatten()

# Ajuste linear simples de escala (0 a 100) com peso de experiência
exp_fator_test = np.clip(X_test_df['anos_experiencia'] / vaga_exp_exigida, 0.2, 1.2)
y_pred_baseline = np.clip(sim_cosseno_test * 100 * exp_fator_test, 0, 100)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
spearman_baseline, _ = spearmanr(y_test, y_pred_baseline)

print('=== Baseline (TF-IDF + Cosine Similarity) ===')
print(f'MAE (Erro Médio Absoluto): {mae_baseline:.2f} pontos')
print(f'RMSE:                     {rmse_baseline:.2f} pontos')
print(f'Correlação de Ranking:    {spearman_baseline:.3f}')

=== Baseline (TF-IDF + Cosine Similarity) ===
MAE (Erro Médio Absoluto): 51.11 pontos
RMSE:                     56.60 pontos
Correlação de Ranking:    0.740


**5. Modelo de Rede Neural (MLPRegressor)**

A rede neural recebe como entradas as featurizações TF-IDF concatenadas com os anos de experiência padronizados, aprendendo a combinação de peso ideal para calcular a aderência.

In [ ]:
# Preparação de features para a Rede Neural
train_linkedin_vec = tfidf.transform(X_train_df['texto_linkedin']).toarray()
test_linkedin_vec = tfidf.transform(X_test_df['texto_linkedin']).toarray()

scaler_exp = StandardScaler()
train_exp_scaled = scaler_exp.fit_transform(X_train_df[['anos_experiencia']])
test_exp_scaled = scaler_exp.transform(X_test_df[['anos_experiencia']])

X_train_nn = np.hstack([train_linkedin_vec, train_exp_scaled])
X_test_nn = np.hstack([test_linkedin_vec, test_exp_scaled])

mlp = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=15
)
mlp.fit(X_train_nn, y_train)

y_pred_mlp = np.clip(mlp.predict(X_test_nn), 0, 100)

mae_mlp = mean_absolute_error(y_test, y_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
spearman_mlp, _ = spearmanr(y_test, y_pred_mlp)

print('=== Rede Neural (MLPRegressor) ===')
print(f'MAE (Erro Médio Absoluto): {mae_mlp:.2f} pontos')
print(f'RMSE:                     {rmse_mlp:.2f} pontos')
print(f'Correlação de Ranking:    {spearman_mlp:.3f}')

=== Rede Neural (MLPRegressor) ===
MAE (Erro Médio Absoluto): 12.62 pontos
RMSE:                     15.42 pontos
Correlação de Ranking:    0.852


**6. Comparação Lado a Lado**

Para a equipe de recrutamento, duas coisas importam: a precisão do score estimado (menor MAE/RMSE) e se a **ordem de classificação (ranking)** dos candidatos está correta (maior Correlação de Spearman).

In [ ]:
comparacao = pd.DataFrame({
    'Baseline (TF-IDF + Regra)': [mae_baseline, rmse_baseline, spearman_baseline],
    'Rede Neural (MLP)': [mae_mlp, rmse_mlp, spearman_mlp]
}, index=['MAE (Menor é melhor)', 'RMSE (Menor é melhor)', 'Ranking Spearman (Maior é melhor)'])

display(comparacao.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(y_test, y_pred_baseline, alpha=0.5, color='#DD8452', label='Baseline')
axes[0].plot([0, 100], [0, 100], 'k--', alpha=0.7)
axes[0].set_xlabel('Match Score Real (%)')
axes[0].set_ylabel('Match Score Previsto (%)')
axes[0].set_title('Baseline: Real vs Previsto')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, y_pred_mlp, alpha=0.5, color='#4C72B0', label='Rede Neural')
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.7)
axes[1].set_xlabel('Match Score Real (%)')
axes[1].set_ylabel('Match Score Previsto (%)')
axes[1].set_title('Rede Neural (MLP): Real vs Previsto')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

                                   Baseline (TF-IDF + Regra)  Rede Neural (MLP)
MAE (Menor é melhor)                                  51.113             12.625
RMSE (Menor é melhor)                                 56.603             15.425
Ranking Spearman (Maior é melhor)                      0.740              0.852


**Como avaliar a assertividade de um modelo de matching na prática?**

Para testar a assertividade de um modelo de recomendação/matching de candidatos em produção, devemos usar **4 dimensões de validação**:

1. **Precision@K (Precisão no Top K):** De todos os K candidatos que o modelo colocou no topo (ex: Top 5 para entrevista), quantos realmente eram adequados segundo os recrutadores humanos?
2. **NDCG (Normalized Discounted Cumulative Gain):** Avalia se os candidatos ideais estão nas **primeiras posições** da lista, penalizando quando um excelente candidato é colocado em posição inferior.
3. **Correlação de Ranking de Spearman:** Mede o grau de concordância entre o ranking gerado pelo modelo e a ordenação real feita por especialistas.
4. **Teste de Aderência Humano (A/B Test):** Apresentar a uma equipe de recrutadores 10 pares de currículos de forma cega (sem saber a nota do modelo) e verificar a taxa de concordância (% de acordo entre a máquina e o especialista).

**Validação com múltiplas seeds**

Para garantir que a superioridade da Rede Neural não é fruto do acaso de uma amostragem específica, testamos o pipeline com 3 seeds (42, 7, 123).

In [ ]:
def rodar_pipeline_matching(seed):
    rng = np.random.RandomState(seed)
    indices_s = rng.choice(len(perfis_base), size=n)
    textos_s = [perfis_base[i] + rng.choice(ruidos_texto) for i in indices_s]
    exp_s = np.round(np.clip(rng.gamma(shape=2.5, scale=1.8, size=n), 0.5, 20.0), 1)
    score_b = np.array([pesos_tecnicos[i] for i in indices_s])
    fator_s = np.where(exp_s < vaga_exp_exigida, (exp_s / vaga_exp_exigida)**1.5, 1.0 + 0.05 * np.log1p(np.maximum(0, exp_s - vaga_exp_exigida)))
    ruido_s = rng.normal(0, 4.5, size=n)
    y_s = np.clip(score_b * fator_s + ruido_s, 0, 100)
    
    df_s = pd.DataFrame({'texto_linkedin': textos_s, 'anos_experiencia': exp_s, 'y': y_s})
    train_s, test_s = train_test_split(df_s, test_size=0.3, random_state=seed)
    
    # TF-IDF
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    vectorizer.fit([vaga_texto] + list(train_s['texto_linkedin']))
    v_vaga = vectorizer.transform([vaga_texto])
    v_test = vectorizer.transform(test_s['texto_linkedin'])
    sim_test = cosine_similarity(v_test, v_vaga).flatten()
    exp_fator = np.clip(test_s['anos_experiencia'] / vaga_exp_exigida, 0.2, 1.2)
    pred_base = np.clip(sim_test * 100 * exp_fator, 0, 100)
    
    # MLP
    v_train = vectorizer.transform(train_s['texto_linkedin']).toarray()
    v_test_arr = v_test.toarray()
    sc_exp = StandardScaler()
    tr_exp = sc_exp.fit_transform(train_s[['anos_experiencia']])
    te_exp = sc_exp.transform(test_s[['anos_experiencia']])
    
    X_tr = np.hstack([v_train, tr_exp])
    X_te = np.hstack([v_test_arr, te_exp])
    
    model = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=seed, early_stopping=True)
    model.fit(X_tr, train_s['y'])
    pred_mlp = np.clip(model.predict(X_te), 0, 100)
    
    return {
        'mae_base': mean_absolute_error(test_s['y'], pred_base),
        'spearman_base': spearmanr(test_s['y'], pred_base)[0],
        'mae_mlp': mean_absolute_error(test_s['y'], pred_mlp),
        'spearman_mlp': spearmanr(test_s['y'], pred_mlp)[0]
    }

seeds = [42, 7, 123]
res_list = [rodar_pipeline_matching(s) for s in seeds]
res_df = pd.DataFrame(res_list, index=[f'Seed {s}' for s in seeds])

summary_df = pd.DataFrame({
    'Baseline (TF-IDF)': [res_df['mae_base'].mean(), res_df['mae_base'].std(), res_df['spearman_base'].mean(), res_df['spearman_base'].std()],
    'Rede Neural (MLP)': [res_df['mae_mlp'].mean(), res_df['mae_mlp'].std(), res_df['spearman_mlp'].mean(), res_df['spearman_mlp'].std()]
}, index=['MAE (média)', 'MAE (desvio-padrão)', 'Spearman (média)', 'Spearman (desvio-padrão)'])

display(summary_df.round(3))

                          Baseline (TF-IDF)  Rede Neural (MLP)
MAE (média)                          49.726              7.420
MAE (desvio-padrão)                   1.237              4.525
Spearman (média)                      0.762              0.934
Spearman (desvio-padrão)              0.019              0.071


**Anexo: Outras Aplicações de Negócio com a Mesma Técnica**

A técnica de matching vetorial e regressão de aderência desenvolvida aqui pode ser estendida para:

### 1. Vendas e B2B
- **Match de Leads e Produtos (Lead Scoring):** Comparar a descrição da empresa cliente com o portfólio de soluções de TI.

### 2. Compras e Suprimentos
- **Matching de Fornecedores:** Comparar os requisitos técnicos de uma RFP/RFI de compra com as propostas e qualificações dos fornecedores cadastradas em catálogo.